In [1]:
# Jupyter cell — Kilosort postprocessing (overwrite-safe) with tidy per-session Excel
# Features:
# - Figures: one per cluster, merged Waveform (µV vs µs) on top + ISI (µs) bottom
# - On waveform: PEAK first, then TROUGh after; dotted vlines + horizontal connector; label Δt (µs)
# - On ISI: 0–1 s (10 ms bins) with modal bin marked
# - Title shows: session | cluster | best channel | average FR (whole recording) | spike count
# - Per-session Excel: TIDY table (one row per cluster, columns are metrics), overwrites old file
# - best_channels.xlsx per results_dir (overwrites old file)
# - Timestamps saved fresh: results_dir/timestamps/*.mat (old timestamps removed)
# - All figures for a session rewritten inside results_dir/figures/{session_name}/

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import scipy.io as sio
from collections import defaultdict
from shutil import rmtree
from kilosort.io import load_ops
from kilosort.data_tools import (
    mean_waveform, cluster_templates, get_best_channel
)

# ---------------------- waveform analysis ----------------------

def analyze_waveform(waveform, fs):
    """
    Peak-first analysis:
      1) global PEAK (max)
      2) TROUGh AFTER the peak (min over waveform[peak_idx:])
      Returns indices/times/values and Peak→Trough Δt in microseconds.
      If peak is last sample, fall back to global trough.
    """
    peak_idx = int(np.argmax(waveform))
    if peak_idx < waveform.size - 1:
        trough_idx = peak_idx + int(np.argmin(waveform[peak_idx:]))
    else:
        trough_idx = int(np.argmin(waveform))
    dt_us = (trough_idx - peak_idx) / fs * 1e6
    return {
        "peak_idx": peak_idx,
        "trough_idx": trough_idx,
        "peak_uV": float(waveform[peak_idx]),
        "trough_uV": float(waveform[trough_idx]),
        "peak_time_us": peak_idx / fs * 1e6,
        "trough_time_us": trough_idx / fs * 1e6,
        "peak_to_trough_us": dt_us
    }

# ---------------------- merged figure (waveform + ISI) ----------------------

def waveform_isi_fig(
    t_us, mean_wv_uV, mean_temp_uV, fs, cluster_id, session_name,
    best_chan, cluster_spikes, all_spikes, save_path_png
):
    """
    Builds a single figure:
      - Top: waveform + template, PEAK (blue) then TROUGh (red), dotted vlines,
             horizontal connector, Δt (µs)
      - Bottom: ISI histogram up to 1 s (x in µs, 10 ms bins), modal bin marked
    Returns metrics dict for Excel.
    """
    # Waveform metrics
    wf = analyze_waveform(mean_wv_uV, fs)
    peak_idx, trough_idx = wf["peak_idx"], wf["trough_idx"]
    dt_us = wf["peak_to_trough_us"]

    # Avg firing rate using WHOLE recording
    n_spikes = int(cluster_spikes.size)
    rec_duration_s = all_spikes.max() / fs if all_spikes.size > 0 else np.nan
    fr_hz = n_spikes / rec_duration_s if rec_duration_s and rec_duration_s > 0 else float('nan')

    # ISI (0–1 s), µs
    isi_us = np.diff(cluster_spikes) / fs * 1e6
    bins_us = np.arange(0, 1_000_000 + 10_000, 10_000)  # 10 ms bins
    counts, edges = np.histogram(isi_us, bins=bins_us)
    if counts.size > 0:
        isi_peak_bin = int(np.argmax(counts))
        isi_peak_center_us = (edges[isi_peak_bin] + edges[isi_peak_bin + 1]) / 2.0
        isi_peak_count = int(counts[isi_peak_bin])
    else:
        isi_peak_center_us = np.nan
        isi_peak_count = 0

    # ---- Figure ----
    fig = plt.figure(figsize=(8, 6))
    gs = fig.add_gridspec(nrows=2, ncols=1, height_ratios=[3, 2], hspace=0.35)

    # Top: Waveform
    ax1 = fig.add_subplot(gs[0, 0])
    ax1.plot(t_us, mean_wv_uV, c='black', linestyle='dashed', linewidth=2, label='Mean waveform')
    ax1.plot(t_us[:mean_temp_uV.shape[-1]], mean_temp_uV, linewidth=1, label='Template')
    ax1.scatter(t_us[peak_idx], mean_wv_uV[peak_idx], c='blue', zorder=5, label='Peak')
    ax1.scatter(t_us[trough_idx], mean_wv_uV[trough_idx], c='red', zorder=5, label='Trough')

    y_min = min(mean_wv_uV.min(), mean_temp_uV.min())
    y_max = max(mean_wv_uV.max(), mean_temp_uV.max())
    ax1.vlines([t_us[peak_idx], t_us[trough_idx]], ymin=y_min, ymax=y_max,
               linestyles='dotted', colors=['blue', 'red'], alpha=0.6)
    y_mid = (mean_wv_uV[peak_idx] + mean_wv_uV[trough_idx]) / 2.0
    ax1.hlines(y=y_mid, xmin=t_us[peak_idx], xmax=t_us[trough_idx],
               linestyles='solid', colors='purple', alpha=0.85)
    ax1.text((t_us[peak_idx] + t_us[trough_idx]) / 2.0, y_mid,
             f"Δt = {dt_us:.1f} µs", ha='center', va='bottom', fontsize=9, color='purple')

    title = (f"{session_name} | Cluster {cluster_id} | Best ch {best_chan} | "
             f"FR {fr_hz:.2f} Hz | n={n_spikes}")
    ax1.set_title(title)
    ax1.set_xlabel("Time (µs)")
    ax1.set_ylabel("Voltage (µV)")
    ax1.legend(loc='best', fontsize=8)

    # Bottom: ISI
    ax2 = fig.add_subplot(gs[1, 0])
    ax2.hist(isi_us, bins=bins_us, edgecolor='none')
    ax2.set_title("ISI Histogram (10 ms bins, up to 1 s)")
    ax2.set_xlabel("ISI (µs)")
    ax2.set_ylabel("Count")
    if np.isfinite(isi_peak_center_us):
        ax2.axvline(isi_peak_center_us, linestyle='--', color='k', alpha=0.8)
        ax2.text(isi_peak_center_us, ax2.get_ylim()[1]*0.9,
                 f"Peak ≈ {int(isi_peak_center_us)} µs\nCount={isi_peak_count}",
                 ha='center', va='top', fontsize=8,
                 bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="gray", alpha=0.75))

    fig.savefig(save_path_png, dpi=150, bbox_inches='tight')
    plt.close(fig)

    return {
        "fr_hz": fr_hz,
        "n_spikes": n_spikes,
        "isi_peak_center_us": isi_peak_center_us,
        "isi_peak_count": isi_peak_count,
        "peak_time_us": wf["peak_time_us"],
        "trough_time_us": wf["trough_time_us"],
        "peak_to_trough_us": wf["peak_to_trough_us"],
        "peak_uV": wf["peak_uV"],
        "trough_uV": wf["trough_uV"]
    }

# ---------------------- per-cluster processing ----------------------

def process_cluster(cluster_id, session_name, results_dir, fs_default=30000):
    results_dir = Path(results_dir)

    # Figures folder per session (already wiped/recreated in top-level)
    figures_dir = results_dir / "figures" / session_name
    figures_dir.mkdir(parents=True, exist_ok=True)

    # Load ops and fs
    fs = fs_default
    try:
        ops = load_ops(results_dir / 'ops.npy')
        fs = float(ops.get('fs', fs_default))
    except Exception:
        pass  # keep default if ops missing/corrupt

    # Waveforms / template on best channel
    mean_wv, spike_subset = mean_waveform(cluster_id, results_dir, n_spikes=100, best=True)
    mean_temp = cluster_templates(cluster_id, results_dir, mean=True, best=True, spike_subset=spike_subset)
    best_chan = get_best_channel(results_dir, cluster_id) + 1  # display 1-based

    # Time axis in µs
    t_us = (np.arange(mean_wv.shape[-1]) / fs) * 1e6

    # Spikes for this cluster and all spikes (for duration)
    spikes = np.load(results_dir / 'spike_times.npy')
    cluster_labels = np.load(results_dir / 'spike_clusters.npy')
    cluster_spikes = np.sort(spikes[cluster_labels == cluster_id])

    # Figure + metrics
    fig_path = figures_dir / f"{session_name}_cluster{cluster_id}.png"
    fig_metrics = waveform_isi_fig(
        t_us=t_us,
        mean_wv_uV=mean_wv,
        mean_temp_uV=mean_temp,
        fs=fs,
        cluster_id=cluster_id,
        session_name=session_name,
        best_chan=best_chan,
        cluster_spikes=cluster_spikes,
        all_spikes=spikes,
        save_path_png=fig_path
    )

    # Save timestamps (.mat) — rebuilt fresh by top-level (timestamps dir wiped)
    ts_dir = results_dir / "timestamps"
    ts_dir.mkdir(exist_ok=True)
    sio.savemat(ts_dir / f"{cluster_id}.mat",
                {f"cluster_{cluster_id}_timestamp": cluster_spikes[:, np.newaxis]})

    # Row for tidy Excel
    return {
        "session": session_name,
        "cluster_id": int(cluster_id),
        "best_channel": int(best_chan),
        "peak_time_us": float(fig_metrics["peak_time_us"]),
        "trough_time_us": float(fig_metrics["trough_time_us"]),
        "peak_to_trough_us": float(fig_metrics["peak_to_trough_us"]),
        "peak_uV": float(fig_metrics["peak_uV"]),
        "trough_uV": float(fig_metrics["trough_uV"]),
        "isi_peak_center_us": float(fig_metrics["isi_peak_center_us"]) if np.isfinite(fig_metrics["isi_peak_center_us"]) else np.nan,
        "isi_peak_count": int(fig_metrics["isi_peak_count"]),
        "avg_firing_rate_Hz": float(fig_metrics["fr_hz"]) if np.isfinite(fig_metrics["fr_hz"]) else np.nan,
        "n_spikes": int(fig_metrics["n_spikes"]),
    }

# ---------------------- top-level: read input Excel & write outputs (tidy, overwrite) ----------------------

def kilosort_postprocess(excel_path):
    """
    Reads an Excel file where each column describes one session:
      row 0: session_name (str)
      row 1: results_dir  (str)
      rows 2+: cluster IDs (ints)
    For each session:
      - Rebuilds figures in {results_dir}/figures/{session_name}/ (old folder removed)
      - Rebuilds timestamps in {results_dir}/timestamps/ (old folder removed)
      - Overwrites per-session tidy Excel -> {results_dir}/{session_name}.xlsx
      - Overwrites per-results_dir best_channels.xlsx with entries from this run
    """
    df = pd.read_excel(excel_path, header=None)
    best_rows_by_dir = defaultdict(list)          # for best_channels.xlsx
    session_rows = defaultdict(list)              # key: (results_dir, session_name) -> list of dicts
    cleaned_results_dirs = set()                  # track which results_dir have been cleaned (timestamps)

    for col in df.columns:
        session_name = str(df.iloc[0, col]).strip()
        results_path = str(df.iloc[1, col]).strip()
        if not results_path:
            continue
        results_dir = Path(results_path)

        # ---- Force-clean folders/files ONCE per session before processing its clusters ----
        # Clean figures/{session_name}
        session_fig_dir = results_dir / "figures" / session_name
        if session_fig_dir.exists():
            rmtree(session_fig_dir)
        session_fig_dir.mkdir(parents=True, exist_ok=True)

        # Clean per-session Excel
        out_xlsx = results_dir / f"{session_name}.xlsx"
        if out_xlsx.exists():
            out_xlsx.unlink()

        # Clean timestamps dir ONCE per results_dir (remove entire folder to avoid stale .mat)
        if results_dir not in cleaned_results_dirs:
            ts_dir = results_dir / "timestamps"
            if ts_dir.exists():
                rmtree(ts_dir)
            ts_dir.mkdir(parents=True, exist_ok=True)
            cleaned_results_dirs.add(results_dir)

        # ---- Process all clusters for this session ----
        cluster_ids = df.iloc[2:, col].dropna()
        for cluster_id in cluster_ids:
            cid = int(cluster_id)
            print(f"Processing {session_name}, Cluster {cid}")
            try:
                row = process_cluster(cid, session_name, results_dir)
                # best_channels accumulation
                best_rows_by_dir[results_dir].append({
                    "session": row["session"],
                    "cluster_id": row["cluster_id"],
                    "best_channel": row["best_channel"]
                })
                # tidy row for this session
                session_rows[(results_dir, session_name)].append(row)
            except Exception as e:
                print(f"Error processing {session_name} Cluster {cid}: {e}")

    # ---- Overwrite best_channels.xlsx per results_dir ----
    for rdir, rows in best_rows_by_dir.items():
        out_bc = Path(rdir) / "best_channels.xlsx"
        if out_bc.exists():
            out_bc.unlink()
        out_df = pd.DataFrame(rows).sort_values(["session", "cluster_id"]).reset_index(drop=True)
        with pd.ExcelWriter(out_bc, engine="openpyxl", mode="w") as writer:
            out_df.to_excel(writer, sheet_name="BestChannels", index=False)
        print(f"[OK] Rewrote {out_bc}")

    # ---- Overwrite tidy per-session Excel ----
    for (rdir, sname), rows in session_rows.items():
        out_df = pd.DataFrame(rows)
        # order columns for convenience
        preferred = [
            "session","cluster_id","best_channel",
            "peak_to_trough_us","peak_time_us","trough_time_us",
            "peak_uV","trough_uV",
            "avg_firing_rate_Hz","n_spikes",
            "isi_peak_center_us","isi_peak_count",
        ]
        cols = [c for c in preferred if c in out_df.columns] + [c for c in out_df.columns if c not in preferred]
        out_df = out_df[cols]

        out_xlsx = Path(rdir) / f"{sname}.xlsx"
        if out_xlsx.exists():
            out_xlsx.unlink()
        with pd.ExcelWriter(out_xlsx, engine="openpyxl", mode="w") as writer:
            out_df.to_excel(writer, sheet_name=sname, index=False)
        print(f"[OK] Rewrote tidy Excel for session {sname}: {out_xlsx}")

    print("Done.")

# --- usage:
# kilosort_postprocess(r"E:\path\to\your\input.xlsx")


In [3]:
feedersheet_dir= Path(r"D:\Data\PTEN Project\feedersheets\single waveforms.xlsx")
kilosort_postprocess(feedersheet_dir)

Processing m30s1, Cluster 14
Processing m30s1, Cluster 29
Processing m30s1, Cluster 30
Processing m30s1, Cluster 40
Processing m30s1, Cluster 44
[OK] Rewrote D:\Data\PTEN Project\PTEN\M30_s1\2024-07-01_13-58-59\binary\kilosort_tl6_tu7_fs30000.0\best_channels.xlsx
[OK] Rewrote tidy Excel for session m30s1: D:\Data\PTEN Project\PTEN\M30_s1\2024-07-01_13-58-59\binary\kilosort_tl6_tu7_fs30000.0\m30s1.xlsx
Done.
